# Complement · Anàlisi univariant

[Obre aquest quadern a Google Colab](https://colab.research.google.com/github/mlacasa/1BATXILLERAT/blob/main/An%C3%A1lisisUnivariante%28I%29.ipynb)

**Matemàtiques · 1r de batxillerat**  
**Autoria: Dr. Lacasa-Cazcarra · Any 2026**

**Objectiu.** Construir taules i gràfiques coherents i interpretar dades numèriques i ordinals.

**Abans de començar:** freqüències, mitjana i mediana  
**Temps orientatiu:** 1–2 sessions. Hi ha activitats bàsiques i ampliacions opcionals.

**Funcionament.** Executa les cel·les en ordre, des del començament. Primer fes una predicció,
després experimenta i escriu una justificació. Els controls són opcionals: també pots
modificar els arguments de les funcions i executar-les. Les figures representen mostres
finites; les propietats infinites necessiten una demostració.

**Procedència.** Reelaboració d'AnálisisUnivariante(I).ipynb; dades simulades amb llavor fixa.


In [ ]:
import math
from fractions import Fraction
from decimal import Decimal, localcontext
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True,
                     "font.size": 11, "figure.dpi": 100})

def explora(funcio, **rangs):
    """Controls opcionals: la mateixa funció també es pot cridar directament."""
    try:
        import ipywidgets as widgets
    except ImportError:
        print("Sense controls: executem els paràmetres per defecte. Pots canviar-los a la crida.")
        return funcio()
    panell = widgets.interactive(funcio, **rangs)
    display(panell)
    return panell


## 1. Dades i freqüències
Una variable discreta pren valors aïllats (no necessàriament enters); una variable
contínua pot prendre qualsevol valor d'un interval en el model matemàtic.
Les dades següents són edats enteres **simulades**, no una mostra real d'alumnes.

Per als intervals usem [a,b), excepte l'últim, que inclou també el màxim.
La taula i l'histograma han de fer servir exactament les mateixes vores.


In [ ]:
import pandas as pd

rng = np.random.default_rng(2026)
edats = rng.integers(18, 66, size=100)

def taula_frequencies(dades, classes=7):
    dades = np.asarray(dades, dtype=float)
    if dades.ndim != 1 or dades.size == 0 or not np.isfinite(dades).all():
        raise ValueError("Cal un vector no buit de nombres finits.")
    if not isinstance(classes, int) or classes < 1:
        raise ValueError("El nombre de classes ha de ser un enter positiu.")
    freq, vores = np.histogram(dades, bins=classes)
    intervals = [f"[{a:.4g}, {b:.4g}{']' if i == len(freq)-1 else ')'}"
                 for i, (a, b) in enumerate(zip(vores[:-1], vores[1:]))]
    taula = pd.DataFrame({"Interval": intervals, "Freqüència": freq,
                          "Freqüència relativa": freq/len(dades),
                          "Acumulada": np.cumsum(freq)})
    return taula, vores

def histograma(classes=7):
    taula, vores = taula_frequencies(edats, classes)
    display(taula)
    fig, ax = plt.subplots()
    ax.hist(edats, bins=vores, edgecolor="white", color="#2563eb")
    ax.set(xlabel="Edat simulada", ylabel="Freqüència", title="Histograma i taula amb els mateixos intervals")
    plt.show()
    print("Total comptat:", int(taula["Freqüència"].sum()), "de", len(edats))
panell = explora(histograma, classes=(1, 15, 1))


## 2. Diagrama de caixa i valors atípics
Separem els conjunts de dades amb noms diferents, perquè executar una activitat
no substitueixi les dades necessàries per a una altra.
Per defecte, els bigotis arriben a les dades més extremes dins de les tanques
$Q_1-1,5\,IQR$ i $Q_3+1,5\,IQR$, amb $IQR=Q_3-Q_1$.
No cal que acabin exactament a les tanques. Un valor atípic no és necessàriament un error.


In [ ]:
rng_caixa = np.random.default_rng(42)
valors = np.concatenate([rng_caixa.normal(50, 10, 100), [10, 110, 115, 120]])
df_valors = pd.DataFrame({"Valors": valors})
df_persones = pd.DataFrame({"Edat": [23, 25, 35, 30, 40], "Salari": [50000, 60000, 80000, 70000, 90000]})
display(df_persones.describe())
print("describe() calcula la desviació estàndard mostral (divisor n−1).")
fig, ax = plt.subplots()
ax.boxplot(df_valors["Valors"])
ax.set(ylabel="Valor simulat", title="Caixa del conjunt df_valors")
plt.show()
q1, q3 = np.quantile(valors, [.25, .75])
iqr = q3-q1
print("Valors fora de les tanques:", valors[(valors < q1-1.5*iqr) | (valors > q3+1.5*iqr)])


## 3. Categories ordinals
En una variable ordinal importa conservar l'ordre de les categories.
Els gràfics han d'utilitzar les dades estudiades, fins i tot si demanem ajuda a una IA
per escriure el codi. No substituïm les observacions per nombres inventats sense indicar-ho.


In [ ]:
ordre = ["Baixa", "Mitjana", "Alta"]
satisfaccio = pd.Series(pd.Categorical(
    ["Baixa", "Mitjana", "Alta", "Baixa", "Mitjana", "Alta", "Alta", "Mitjana", "Baixa", "Mitjana"],
    categories=ordre, ordered=True))
recomptes = satisfaccio.value_counts(sort=False).reindex(ordre, fill_value=0)
fig, ax = plt.subplots()
ax.barh(ordre, recomptes.to_numpy(), color=["#2563eb", "#16a34a", "#eab308"])
ax.set(xlabel="Freqüència", title="Satisfacció: les dades originals de l'activitat")
plt.show()


## Activitats
1. Canvia el nombre de classes. Canvien les dades o només l'agrupació?
2. Executa `taula_frequencies([18,19,20], 2)` i `taula_frequencies([20,20,20], 2)`.
   Comprova que sempre es compten les tres observacions.
3. Per què no podem concloure que tots els valors atípics són errors de mesura?
4. Redacta una interpretació del gràfic ordinal que coincideixi amb els recomptes.

<details><summary>Comprovació</summary>

Canvia l'agrupació, no les dades. Les freqüències ordinals són 3, 4 i 3.
Cal investigar el context abans de decidir si un valor atípic és un error.
</details>


**Navegació:** [Índex i guia](README.md)
